# PRISM Indexer Brain Fine-Tuning

Fine-tune **Qwen3.5** on materials science NL → Cypher translation using **Unsloth**.

| GPU | Model |
|-----|-------|
| T4 16GB (free Colab) | Qwen3.5-4B |
| A100 40GB | Qwen3.5-9B |
| A100 80GB | Qwen3.5-27B |

**Output:** GGUF file you load into Ollama on your PRISM node.

Press **Runtime → Run all** on a Colab GPU instance.

## 1. Installation

In [1]:
%%capture
import os, importlib.util

# Install order matters: transformers v5 first, then unsloth latest
!pip install --upgrade --no-cache-dir transformers>=5.0 trl
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

# Verify versions
import transformers, unsloth
print(f"transformers: {transformers.__version__}")
print(f"unsloth: {unsloth.__version__}")

## 2. Configuration

Change `MODEL_SIZE` based on your GPU. Everything else auto-adjusts.

In [2]:
# ============================================================
# CONFIG — edit this cell
# ============================================================
MODEL_SIZE = "9B"  # "4B" for T4, "9B" for A100-40, "27B" for A100-80

MODEL_MAP = {
    "4B":  "Qwen/Qwen3.5-4B",
    "9B":  "Qwen/Qwen3.5-9B",
    "27B": "Qwen/Qwen3.5-27B",
}
MODEL_NAME = MODEL_MAP[MODEL_SIZE]

MAX_SEQ_LENGTH = 2048
LORA_RANK      = 16
LORA_ALPHA     = 16
BATCH_SIZE     = 1
GRAD_ACCUM     = 4
LR             = 2e-4
MAX_STEPS      = 500       # increase for production run
WARMUP_STEPS   = 20
OUTPUT_DIR     = "prism-indexer-lora"
GGUF_DIR       = "prism-indexer-gguf"
GGUF_QUANT     = "q4_k_m"  # q4_k_m for 24GB local, q8_0 for more VRAM

# Optional: push to HuggingFace
HF_REPO = None  # e.g. "marc27/prism-indexer-4b-gguf"
HF_TOKEN = None  # or set via Colab Secrets

print(f"Model: {MODEL_NAME}  |  LoRA r={LORA_RANK}  |  Steps: {MAX_STEPS}")

Model: Qwen/Qwen3.5-9B  |  LoRA r=16  |  Steps: 500


## 3. Load Model

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # QLoRA NOT recommended for Qwen3.5
    load_in_16bit=True,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f"Trainable params: {model.print_trainable_parameters()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


NameError: name 'auto_docstring' is not defined

## 4. PRISM Materials Science Dataset

This section defines the training data: natural language questions about materials
paired with correct Cypher queries for the PRISM knowledge graph.

### Graph Schema

```
Node labels:        Alloy, Element, Property, Process, Phase, Paper, Author, Dataset
Relationship types: CONTAINS, HAS_PROPERTY, PROCESSED_BY, OBSERVED_IN,
                    PUBLISHED_IN, AUTHORED_BY, PART_OF, CITES

Patterns:
  (alloy:Alloy)-[:CONTAINS {weight: 0.25}]->(element:Element)
  (alloy:Alloy)-[:HAS_PROPERTY]->(prop:Property {value: 1200, unit: 'MPa'})
  (alloy:Alloy)-[:PROCESSED_BY {order: 1}]->(process:Process)
  (alloy:Alloy)-[:OBSERVED_IN]->(phase:Phase)
  (paper:Paper)-[:AUTHORED_BY]->(author:Author)
  (paper:Paper)-[:CITES]->(paper2:Paper)
```

In [ ]:
# ============================================================
# Training examples: NL question -> Cypher query
# ============================================================
# Add your own domain-specific examples here. More examples = better model.
# Aim for 200-500+ examples for a production fine-tune.

TRAINING_PAIRS = [
    # --- Composition queries ---
    {
        "question": "What elements are in NbMoTaW?",
        "cypher": "MATCH (a:Alloy {name: 'NbMoTaW'})-[:CONTAINS]->(e:Element) RETURN e.name AS element, r.weight AS fraction LIMIT 20",
        "explanation": "Finds all elements contained in the NbMoTaW alloy with their weight fractions."
    },
    {
        "question": "Which alloys contain tungsten?",
        "cypher": "MATCH (a:Alloy)-[:CONTAINS]->(e:Element) WHERE toLower(e.name) CONTAINS toLower('W') OR toLower(e.name) CONTAINS toLower('tungsten') RETURN a.name AS alloy LIMIT 20",
        "explanation": "Lists alloys that contain the element tungsten (W)."
    },
    {
        "question": "Find alloys with more than 20% chromium",
        "cypher": "MATCH (a:Alloy)-[r:CONTAINS]->(e:Element) WHERE toLower(e.name) CONTAINS toLower('Cr') AND r.weight > 0.20 RETURN a.name AS alloy, r.weight AS cr_fraction LIMIT 20",
        "explanation": "Finds alloys where chromium content exceeds 20% by weight."
    },
    {
        "question": "What is the composition of Inconel 718?",
        "cypher": "MATCH (a:Alloy)-[r:CONTAINS]->(e:Element) WHERE toLower(a.name) CONTAINS toLower('Inconel 718') RETURN e.name AS element, r.weight AS fraction ORDER BY r.weight DESC LIMIT 20",
        "explanation": "Returns all elements and their weight fractions in Inconel 718, sorted by abundance."
    },
    {
        "question": "Which alloys have both nickel and iron?",
        "cypher": "MATCH (a:Alloy)-[:CONTAINS]->(ni:Element), (a)-[:CONTAINS]->(fe:Element) WHERE toLower(ni.name) IN ['ni', 'nickel'] AND toLower(fe.name) IN ['fe', 'iron'] RETURN a.name AS alloy LIMIT 20",
        "explanation": "Finds alloys containing both nickel and iron."
    },

    # --- Property queries ---
    {
        "question": "What is the hardness of Ti-6Al-4V?",
        "cypher": "MATCH (a:Alloy)-[:HAS_PROPERTY]->(p:Property) WHERE toLower(a.name) CONTAINS toLower('Ti-6Al-4V') AND toLower(p.name) CONTAINS toLower('hardness') RETURN p.value AS hardness, p.unit AS unit LIMIT 5",
        "explanation": "Retrieves the hardness property of the Ti-6Al-4V alloy."
    },
    {
        "question": "Find alloys with tensile strength above 1000 MPa",
        "cypher": "MATCH (a:Alloy)-[:HAS_PROPERTY]->(p:Property) WHERE toLower(p.name) CONTAINS toLower('tensile strength') AND p.value > 1000 AND toLower(p.unit) = 'mpa' RETURN a.name AS alloy, p.value AS strength_mpa ORDER BY p.value DESC LIMIT 20",
        "explanation": "Lists alloys with tensile strength exceeding 1000 MPa, sorted highest first."
    },
    {
        "question": "Compare yield strength of all titanium alloys",
        "cypher": "MATCH (a:Alloy)-[:CONTAINS]->(e:Element), (a)-[:HAS_PROPERTY]->(p:Property) WHERE toLower(e.name) IN ['ti', 'titanium'] AND toLower(p.name) CONTAINS toLower('yield strength') RETURN DISTINCT a.name AS alloy, p.value AS yield_strength, p.unit AS unit ORDER BY p.value DESC LIMIT 20",
        "explanation": "Compares yield strength across all titanium-containing alloys."
    },
    {
        "question": "Which alloys have melting point above 2000 degrees?",
        "cypher": "MATCH (a:Alloy)-[:HAS_PROPERTY]->(p:Property) WHERE toLower(p.name) CONTAINS toLower('melting') AND p.value > 2000 RETURN a.name AS alloy, p.value AS melting_point, p.unit AS unit ORDER BY p.value DESC LIMIT 20",
        "explanation": "Finds alloys with melting points exceeding 2000 degrees."
    },
    {
        "question": "What properties does Haynes 230 have?",
        "cypher": "MATCH (a:Alloy)-[:HAS_PROPERTY]->(p:Property) WHERE toLower(a.name) CONTAINS toLower('Haynes 230') RETURN p.name AS property, p.value AS value, p.unit AS unit LIMIT 20",
        "explanation": "Lists all recorded properties for Haynes 230."
    },

    # --- Processing queries ---
    {
        "question": "How is NbMoTaW processed?",
        "cypher": "MATCH (a:Alloy {name: 'NbMoTaW'})-[r:PROCESSED_BY]->(p:Process) RETURN p.name AS step, r.order AS sequence ORDER BY r.order LIMIT 20",
        "explanation": "Returns the processing steps for NbMoTaW in order."
    },
    {
        "question": "Which alloys use arc melting?",
        "cypher": "MATCH (a:Alloy)-[:PROCESSED_BY]->(p:Process) WHERE toLower(p.name) CONTAINS toLower('arc melting') RETURN a.name AS alloy LIMIT 20",
        "explanation": "Lists alloys that undergo arc melting in their processing route."
    },
    {
        "question": "Find alloys processed by both sintering and HIP",
        "cypher": "MATCH (a:Alloy)-[:PROCESSED_BY]->(s:Process), (a)-[:PROCESSED_BY]->(h:Process) WHERE toLower(s.name) CONTAINS toLower('sinter') AND toLower(h.name) CONTAINS toLower('hip') RETURN a.name AS alloy LIMIT 20",
        "explanation": "Finds alloys whose processing route includes both sintering and hot isostatic pressing."
    },

    # --- Phase queries ---
    {
        "question": "What phases are observed in CoCrFeMnNi?",
        "cypher": "MATCH (a:Alloy)-[:OBSERVED_IN]->(ph:Phase) WHERE toLower(a.name) CONTAINS toLower('CoCrFeMnNi') RETURN ph.name AS phase LIMIT 20",
        "explanation": "Lists crystallographic phases observed in the Cantor alloy CoCrFeMnNi."
    },
    {
        "question": "Which alloys form BCC phase?",
        "cypher": "MATCH (a:Alloy)-[:OBSERVED_IN]->(ph:Phase) WHERE toLower(ph.name) CONTAINS toLower('bcc') RETURN a.name AS alloy LIMIT 20",
        "explanation": "Lists alloys in which a BCC phase has been observed."
    },

    # --- Literature queries ---
    {
        "question": "Find papers about high entropy alloys",
        "cypher": "MATCH (p:Paper) WHERE toLower(p.name) CONTAINS toLower('high entropy') RETURN p.name AS title LIMIT 20",
        "explanation": "Searches for papers with 'high entropy' in the title."
    },
    {
        "question": "Who authored the most papers on refractory alloys?",
        "cypher": "MATCH (p:Paper)-[:AUTHORED_BY]->(a:Author) WHERE toLower(p.name) CONTAINS toLower('refractory') RETURN a.name AS author, count(p) AS paper_count ORDER BY paper_count DESC LIMIT 10",
        "explanation": "Ranks authors by number of papers mentioning refractory alloys."
    },
    {
        "question": "What papers cite Senkov 2011?",
        "cypher": "MATCH (p:Paper)-[:CITES]->(cited:Paper) WHERE toLower(cited.name) CONTAINS toLower('senkov') AND cited.year = 2011 RETURN p.name AS citing_paper LIMIT 20",
        "explanation": "Finds all papers that cite a Senkov 2011 publication."
    },

    # --- Cross-domain queries ---
    {
        "question": "Find alloys with high hardness that contain tungsten",
        "cypher": "MATCH (a:Alloy)-[:CONTAINS]->(e:Element), (a)-[:HAS_PROPERTY]->(p:Property) WHERE toLower(e.name) IN ['w', 'tungsten'] AND toLower(p.name) CONTAINS toLower('hardness') RETURN a.name AS alloy, p.value AS hardness, p.unit AS unit ORDER BY p.value DESC LIMIT 20",
        "explanation": "Finds tungsten-containing alloys ranked by hardness."
    },
    {
        "question": "Show me refractory high-entropy alloys with BCC structure",
        "cypher": "MATCH (a:Alloy)-[:OBSERVED_IN]->(ph:Phase), (a)-[:CONTAINS]->(e:Element) WHERE toLower(ph.name) CONTAINS toLower('bcc') AND e.name IN ['Nb', 'Mo', 'Ta', 'W', 'V', 'Hf', 'Zr', 'Ti', 'Cr'] WITH a, count(DISTINCT e) AS num_elements WHERE num_elements >= 4 RETURN DISTINCT a.name AS alloy, num_elements LIMIT 20",
        "explanation": "Finds alloys with BCC phase containing 4+ refractory elements (high-entropy candidates)."
    },
    {
        "question": "Which datasets contain data about nickel superalloys?",
        "cypher": "MATCH (a:Alloy)-[:PART_OF]->(d:Dataset), (a)-[:CONTAINS]->(e:Element) WHERE toLower(e.name) IN ['ni', 'nickel'] RETURN DISTINCT d.name AS dataset, count(a) AS alloy_count ORDER BY alloy_count DESC LIMIT 20",
        "explanation": "Lists datasets that contain nickel-based alloys, ranked by count."
    },

    # --- Aggregation queries ---
    {
        "question": "How many alloys are in the database?",
        "cypher": "MATCH (a:Alloy) RETURN count(a) AS total_alloys",
        "explanation": "Counts the total number of alloy nodes in the graph."
    },
    {
        "question": "What are the most common elements across all alloys?",
        "cypher": "MATCH (a:Alloy)-[:CONTAINS]->(e:Element) RETURN e.name AS element, count(a) AS alloy_count ORDER BY alloy_count DESC LIMIT 20",
        "explanation": "Ranks elements by how many alloys contain them."
    },
    {
        "question": "Average tensile strength of iron-based alloys",
        "cypher": "MATCH (a:Alloy)-[:CONTAINS]->(e:Element), (a)-[:HAS_PROPERTY]->(p:Property) WHERE toLower(e.name) IN ['fe', 'iron'] AND toLower(p.name) CONTAINS toLower('tensile strength') RETURN avg(p.value) AS avg_tensile, p.unit AS unit LIMIT 1",
        "explanation": "Calculates average tensile strength across iron-containing alloys."
    },
    {
        "question": "List all element names in the graph",
        "cypher": "MATCH (e:Element) RETURN e.name AS element ORDER BY e.name LIMIT 200",
        "explanation": "Returns all element nodes alphabetically."
    },
    {
        "question": "Show alloys sorted by number of elements",
        "cypher": "MATCH (a:Alloy)-[:CONTAINS]->(e:Element) RETURN a.name AS alloy, count(e) AS num_elements ORDER BY num_elements DESC LIMIT 20",
        "explanation": "Ranks alloys by compositional complexity (element count)."
    },

    # --- Negative / safety examples ---
    {
        "question": "Delete all alloys from the database",
        "cypher": "MATCH (a:Alloy) RETURN a.name AS alloy LIMIT 20",
        "explanation": "Read-only query listing alloys. Write operations like DELETE are not permitted."
    },
    {
        "question": "Drop everything in the graph",
        "cypher": "MATCH (n) RETURN labels(n)[0] AS type, count(n) AS count LIMIT 20",
        "explanation": "Read-only summary of graph contents by node type. Destructive operations are not allowed."
    },
]

print(f"Training examples: {len(TRAINING_PAIRS)}")

### 4a. Generate Training Data from MARC27 Knowledge API

Pulls real entities from the MARC27 platform (200K+ nodes, 6M+ edges)
and generates NL→Cypher training pairs automatically.

**Requirements:** `pip install marc27` + valid API token (`prism login`).
If the API is unavailable, this cell is skipped and the 27 built-in examples above are used.

In [ ]:
# ============================================================
# Auto-generate training pairs from MARC27 Knowledge Graph API
# ============================================================
import random
random.seed(42)

MARC27_AVAILABLE = False
try:
    from marc27 import PlatformClient
    client = PlatformClient()
    stats = client.knowledge.graph_stats()
    print(f"Connected to MARC27 Knowledge Graph: {stats.get('nodes', 0):,} nodes, {stats.get('edges', 0):,} edges")
    MARC27_AVAILABLE = True
except Exception as e:
    print(f"MARC27 API not available ({e}). Using built-in examples only.")

if MARC27_AVAILABLE:
    generated_pairs = []

    # --- Fetch real entity samples from the API ---
    ELEMENT_QUERIES = ["titanium", "nickel", "iron", "chromium", "aluminum",
                       "tungsten", "molybdenum", "cobalt", "vanadium", "niobium",
                       "copper", "manganese", "silicon", "zirconium", "hafnium"]
    ALLOY_QUERIES = ["inconel", "hastelloy", "haynes", "waspaloy", "Ti-6Al",
                     "stainless", "cantor", "NbMoTaW", "CoCrFeNi", "AlCoCrFeNi",
                     "superalloy", "steel", "bronze", "monel", "nimonic"]
    PROPERTY_QUERIES = ["hardness", "tensile strength", "yield strength",
                        "melting point", "density", "elastic modulus",
                        "thermal conductivity", "elongation", "fracture toughness"]
    PROCESS_QUERIES = ["arc melting", "sintering", "hot isostatic pressing",
                       "annealing", "cold rolling", "powder metallurgy",
                       "spark plasma sintering", "casting", "forging"]
    PHASE_QUERIES = ["BCC", "FCC", "HCP", "sigma", "laves", "B2", "L12"]

    # Collect real entity names from the API
    real_alloys = set()
    real_elements = set()
    real_properties = set()
    real_processes = set()
    real_phases = set()

    print("Fetching entities from MARC27...")
    for q in ALLOY_QUERIES:
        try:
            results = client.knowledge.graph_search(q, limit=20)
            for r in results:
                if r.get("type") == "Alloy" and r.get("name"):
                    real_alloys.add(r["name"])
        except: pass

    for q in ELEMENT_QUERIES:
        try:
            results = client.knowledge.graph_search(q, limit=10)
            for r in results:
                if r.get("type") == "Element" and r.get("name"):
                    real_elements.add(r["name"])
        except: pass

    for q in PROPERTY_QUERIES:
        try:
            results = client.knowledge.graph_search(q, limit=10)
            for r in results:
                if r.get("type") == "Property" and r.get("name"):
                    real_properties.add(r["name"])
        except: pass

    for q in PROCESS_QUERIES:
        try:
            results = client.knowledge.graph_search(q, limit=10)
            for r in results:
                if r.get("type") == "Process" and r.get("name"):
                    real_processes.add(r["name"])
        except: pass

    for q in PHASE_QUERIES:
        try:
            results = client.knowledge.graph_search(q, limit=10)
            for r in results:
                if r.get("type") == "Phase" and r.get("name"):
                    real_phases.add(r["name"])
        except: pass

    real_alloys = sorted(real_alloys)
    real_elements = sorted(real_elements)
    real_properties = sorted(real_properties)
    real_processes = sorted(real_processes)
    real_phases = sorted(real_phases)

    print(f"  Alloys: {len(real_alloys)}, Elements: {len(real_elements)}, "
          f"Properties: {len(real_properties)}, Processes: {len(real_processes)}, "
          f"Phases: {len(real_phases)}")

    # --- Generate NL->Cypher pairs from real entities ---

    # Composition queries
    for alloy in real_alloys[:50]:
        generated_pairs.append({
            "question": f"What elements are in {alloy}?",
            "cypher": f"MATCH (a:Alloy)-[r:CONTAINS]->(e:Element) WHERE toLower(a.name) CONTAINS toLower('{alloy}') RETURN e.name AS element, r.weight AS fraction ORDER BY r.weight DESC LIMIT 20",
            "explanation": f"Returns the elemental composition of {alloy}."
        })

    for elem in real_elements[:30]:
        generated_pairs.append({
            "question": f"Which alloys contain {elem}?",
            "cypher": f"MATCH (a:Alloy)-[:CONTAINS]->(e:Element) WHERE toLower(e.name) CONTAINS toLower('{elem}') RETURN a.name AS alloy LIMIT 20",
            "explanation": f"Lists alloys containing the element {elem}."
        })

    # Multi-element queries
    if len(real_elements) >= 2:
        for _ in range(30):
            e1, e2 = random.sample(real_elements[:20], 2)
            generated_pairs.append({
                "question": f"Find alloys with both {e1} and {e2}",
                "cypher": f"MATCH (a:Alloy)-[:CONTAINS]->(e1:Element), (a)-[:CONTAINS]->(e2:Element) WHERE toLower(e1.name) CONTAINS toLower('{e1}') AND toLower(e2.name) CONTAINS toLower('{e2}') RETURN a.name AS alloy LIMIT 20",
                "explanation": f"Finds alloys containing both {e1} and {e2}."
            })

    # Property queries for real alloys
    for alloy in random.sample(real_alloys[:40], min(30, len(real_alloys))):
        generated_pairs.append({
            "question": f"What properties does {alloy} have?",
            "cypher": f"MATCH (a:Alloy)-[:HAS_PROPERTY]->(p:Property) WHERE toLower(a.name) CONTAINS toLower('{alloy}') RETURN p.name AS property, p.value AS value, p.unit AS unit LIMIT 20",
            "explanation": f"Lists all recorded properties for {alloy}."
        })

    # Specific property lookups
    for prop in real_properties[:15]:
        for alloy in random.sample(real_alloys[:30], min(3, len(real_alloys))):
            generated_pairs.append({
                "question": f"What is the {prop} of {alloy}?",
                "cypher": f"MATCH (a:Alloy)-[:HAS_PROPERTY]->(p:Property) WHERE toLower(a.name) CONTAINS toLower('{alloy}') AND toLower(p.name) CONTAINS toLower('{prop}') RETURN p.value AS value, p.unit AS unit LIMIT 5",
                "explanation": f"Retrieves the {prop} of {alloy}."
            })

    # Processing queries
    for alloy in random.sample(real_alloys[:30], min(20, len(real_alloys))):
        generated_pairs.append({
            "question": f"How is {alloy} processed?",
            "cypher": f"MATCH (a:Alloy)-[r:PROCESSED_BY]->(p:Process) WHERE toLower(a.name) CONTAINS toLower('{alloy}') RETURN p.name AS step, r.order AS sequence ORDER BY r.order LIMIT 20",
            "explanation": f"Returns the processing steps for {alloy}."
        })

    for proc in real_processes[:10]:
        generated_pairs.append({
            "question": f"Which alloys are made by {proc}?",
            "cypher": f"MATCH (a:Alloy)-[:PROCESSED_BY]->(p:Process) WHERE toLower(p.name) CONTAINS toLower('{proc}') RETURN a.name AS alloy LIMIT 20",
            "explanation": f"Lists alloys processed by {proc}."
        })

    # Phase queries
    for alloy in random.sample(real_alloys[:30], min(15, len(real_alloys))):
        generated_pairs.append({
            "question": f"What phases does {alloy} have?",
            "cypher": f"MATCH (a:Alloy)-[:OBSERVED_IN]->(ph:Phase) WHERE toLower(a.name) CONTAINS toLower('{alloy}') RETURN ph.name AS phase LIMIT 20",
            "explanation": f"Lists crystallographic phases observed in {alloy}."
        })

    for phase in real_phases[:10]:
        generated_pairs.append({
            "question": f"Which alloys have {phase} phase?",
            "cypher": f"MATCH (a:Alloy)-[:OBSERVED_IN]->(ph:Phase) WHERE toLower(ph.name) CONTAINS toLower('{phase}') RETURN a.name AS alloy LIMIT 20",
            "explanation": f"Lists alloys with observed {phase} phase."
        })

    # Neighbor exploration (from graph_entity API)
    explored = 0
    for alloy in random.sample(real_alloys[:20], min(10, len(real_alloys))):
        try:
            entity = client.knowledge.graph_entity(alloy, limit=5)
            neighbors = entity.get("neighbors", {})
            for rel_type, targets in neighbors.items():
                if not targets: continue
                target_name = targets[0].get("name", "") if isinstance(targets[0], dict) else str(targets[0])
                if not target_name: continue
                generated_pairs.append({
                    "question": f"How is {alloy} related to {target_name}?",
                    "cypher": f"MATCH (a)-[r]->(b) WHERE toLower(a.name) CONTAINS toLower('{alloy}') AND toLower(b.name) CONTAINS toLower('{target_name}') RETURN type(r) AS relationship, a.name AS from_entity, b.name AS to_entity LIMIT 10",
                    "explanation": f"Shows the relationship between {alloy} and {target_name}."
                })
                explored += 1
        except: pass

    # Path queries between random entity pairs
    if len(real_alloys) >= 2:
        for _ in range(15):
            a1, a2 = random.sample(real_alloys[:20], 2)
            generated_pairs.append({
                "question": f"How are {a1} and {a2} related?",
                "cypher": f"MATCH path = shortestPath((a:Alloy)-[*..3]-(b:Alloy)) WHERE toLower(a.name) CONTAINS toLower('{a1}') AND toLower(b.name) CONTAINS toLower('{a2}') RETURN [n IN nodes(path) | n.name] AS path_nodes, [r IN relationships(path) | type(r)] AS path_rels LIMIT 5",
                "explanation": f"Finds the shortest path connecting {a1} and {a2} in the knowledge graph."
            })

    # Merge with built-in pairs
    TRAINING_PAIRS.extend(generated_pairs)
    print(f"\nGenerated {len(generated_pairs)} pairs from MARC27 API")
    print(f"Total training pairs: {len(TRAINING_PAIRS)}")

    # Save to JSON for reuse
    os.makedirs("data", exist_ok=True)
    with open("data/indexer_training.json", "w") as f:
        json.dump(TRAINING_PAIRS, f, indent=2)
    print(f"Saved to data/indexer_training.json")

### 4b. Format as Chat Dataset

Converts NL/Cypher pairs into the chat format Qwen expects.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a Neo4j Cypher query expert for a materials science knowledge graph.

## Graph Schema
Node labels: [Alloy, Element, Property, Process, Phase, Paper, Author, Dataset]
Relationship types: [CONTAINS, HAS_PROPERTY, PROCESSED_BY, OBSERVED_IN, PUBLISHED_IN, AUTHORED_BY, PART_OF, CITES]

All nodes have a `name` property. Common patterns:
- (alloy:Alloy)-[:CONTAINS {weight}]->(element:Element)
- (alloy:Alloy)-[:HAS_PROPERTY]->(prop:Property {value, unit})
- (alloy:Alloy)-[:PROCESSED_BY {order}]->(process:Process)
- (alloy:Alloy)-[:OBSERVED_IN]->(phase:Phase)
- (paper:Paper)-[:AUTHORED_BY]->(author:Author)

## Rules
1. Return ONLY valid Neo4j Cypher
2. Always include LIMIT unless user asks for all results
3. Use toLower() for case-insensitive text matching
4. Return useful columns (name, type, values) not raw node objects
5. NEVER generate write operations (DELETE, CREATE, MERGE, SET, REMOVE, DROP)

## Response Format
CYPHER: <your cypher query>
EXPLANATION: <one sentence>"""


def format_example(pair):
    """Convert a training pair into Qwen chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": pair["question"]},
        {"role": "assistant", "content": f"CYPHER: {pair['cypher']}\nEXPLANATION: {pair['explanation']}"},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}


dataset = Dataset.from_list(TRAINING_PAIRS)
dataset = dataset.map(format_example)

print(f"Dataset size: {len(dataset)}")
print(f"\n--- Example formatted text ---\n{dataset[0]['text'][:500]}...")

### 4c. (Optional) Load Additional Examples from JSON

If you have a larger dataset at `data/indexer_training.json`, load it here.
Expected format: `[{"question": "...", "cypher": "...", "explanation": "..."}]`

In [ ]:
import json, os

EXTRA_DATA = "data/indexer_training.json"  # set to None to skip

if EXTRA_DATA and os.path.exists(EXTRA_DATA):
    with open(EXTRA_DATA) as f:
        extra_pairs = json.load(f)
    extra_ds = Dataset.from_list(extra_pairs).map(format_example)
    from datasets import concatenate_datasets
    dataset = concatenate_datasets([dataset, extra_ds])
    print(f"Extended dataset size: {len(dataset)}")
else:
    print("No extra data file found, using built-in examples only.")

## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        max_steps=MAX_STEPS,
        learning_rate=LR,
        logging_steps=5,
        output_dir=OUTPUT_DIR,
        optim="adamw_8bit",
        seed=3407,
        dataset_num_proc=1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

stats = trainer.train()
print(f"\nTraining loss: {stats.training_loss:.4f}")

## 6. Test the Model

In [ ]:
# Quick inference test
FastLanguageModel.for_inference(model)

test_questions = [
    "What elements are in CoCrFeMnNi?",
    "Find alloys with yield strength above 800 MPa",
    "Which alloys use powder metallurgy?",
    "How many papers mention refractory high-entropy alloys?",
    "Delete the database",  # should refuse
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        input_ids=inputs, max_new_tokens=256, temperature=0.1, top_p=0.9,
    )
    response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nQ: {q}")
    print(f"A: {response.strip()}")
    print("-" * 60)

## 7. Save LoRA Adapters

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapters saved to {OUTPUT_DIR}/")

## 8. Export GGUF for Ollama

This creates a quantized GGUF file you can load directly into Ollama on your PRISM node:

```bash
# On your local machine after downloading the GGUF:
ollama create prism-indexer -f Modelfile
```

Where `Modelfile` contains:
```
FROM ./prism-indexer-gguf/unsloth.Q4_K_M.gguf
```

In [ ]:
model.save_pretrained_gguf(
    GGUF_DIR, tokenizer, quantization_method=GGUF_QUANT
)
print(f"GGUF saved to {GGUF_DIR}/ ({GGUF_QUANT})")

# Show file size
import glob
for f in glob.glob(f"{GGUF_DIR}/*.gguf"):
    size_gb = os.path.getsize(f) / (1024**3)
    print(f"  {f}: {size_gb:.1f} GB")

## 9. (Optional) Push to HuggingFace

In [ ]:
if HF_REPO and HF_TOKEN:
    model.push_to_hub_gguf(
        HF_REPO, tokenizer,
        quantization_method=GGUF_QUANT,
        token=HF_TOKEN,
    )
    print(f"Pushed to https://huggingface.co/{HF_REPO}")
else:
    print("Skipping HF push (set HF_REPO and HF_TOKEN to enable).")
    print(f"Download the GGUF manually: Files > {GGUF_DIR}/")

## 10. Deploy to PRISM

After downloading the GGUF file to your local machine:

```bash
# 1. Create Ollama model
cat > Modelfile << 'EOF'
FROM ./prism-indexer-gguf/unsloth.Q4_K_M.gguf
PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "</s>"
EOF

ollama create prism-indexer -f Modelfile

# 2. Test it
ollama run prism-indexer 'What elements are in NbMoTaW?'

# 3. Configure PRISM to use it
# In ~/.prism/settings.json:
# { "indexer": { "model": "prism-indexer" } }
```

The `NlQueryTranslator` in `crates/ingest/src/nl_query.rs` will call
this model via Ollama's `/api/generate` endpoint automatically.